In [ ]:
########################################################################
# The script is used to merge multiple NetCDF files and regrid the data.
########################################################################

# Import necessary libraries
import glob
import os

import numpy as np
import pandas as pd

import xarray as xr
import xesmf as xe

import cftime

import subprocess

import warnings; warnings.filterwarnings('ignore')


In [ ]:
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
def merge_cmip6_files(
    variable, experiment, source_id, table_id, project="CMIP6", verbose=False
):
    """
    Merge and standardize multiple NetCDF files.
    """
    # Data directory
    data_dir = "/Users/hao/EXTERNAL/DATA/CMIP"
    search_dir = os.path.join(
        data_dir, project, experiment, table_id, variable, source_id
    )

    if verbose:
        print(f"Search directory: {search_dir}")

    # Find all NetCDF files
    nc_files = sorted(glob.glob(os.path.join(search_dir, "*.nc")))

    if not nc_files:
        if verbose:
            print(f"No files found for {variable}")
        return None

    if verbose:
        print(f"Found {len(nc_files)} files")

    try:
        datasets = []
        for i, f in enumerate(nc_files):

            # Load variable data
            ds = xr.open_dataset(f, chunks={"time": 12})[variable]

            # Standardize coordinates
            ds = _standardize_coordinates(ds, verbose=(verbose and i == 0))

            datasets.append(ds)

        # Merge all datasets
        merged_ds = xr.concat(datasets, dim="time", compat="override", coords="minimal")

        if verbose:
            print(
                f"Merged {len(datasets)} files, time range: {merged_ds.time.values[0]} to {merged_ds.time.values[-1]}"
            )

        return merged_ds

    except Exception as e:
        if verbose:
            print(f"Error merging files: {e}")
        return None

    finally:
        # Clean up
        for ds in datasets:
            try:
                ds.close()
            except:
                pass


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
def _standardize_coordinates(ds, verbose=False):
    """
    Coordinate standardization function.
    """
    # Find spatial dimensions
    lat_dim = next(
        (dim for dim in ["lat", "latitude", "y", "nlat"] if dim in ds.dims), None
    )
    lon_dim = next(
        (dim for dim in ["lon", "longitude", "x", "nlon"] if dim in ds.dims), None
    )

    if not (lat_dim and lon_dim):
        return ds

    coords_to_drop = []
    for coord_name in ["nav_lat", "nav_lon", "lat", "lon", "latitude", "longitude"]:
        if coord_name in ds.coords and ds.coords[coord_name].ndim == 2:
            coords_to_drop.append(coord_name)

    if coords_to_drop:
        ds = ds.drop_vars(coords_to_drop)

    rename_dict = {}
    if lat_dim != "lat":
        rename_dict[lat_dim] = "lat"
    if lon_dim != "lon":
        rename_dict[lon_dim] = "lon"

    if rename_dict:
        ds = ds.rename(rename_dict)

    if "lat" not in ds.coords:
        lat_size = ds.sizes["lat"]
        ds = ds.assign_coords(lat=np.linspace(-90, 90, lat_size))

    if "lon" not in ds.coords:
        lon_size = ds.sizes["lon"]
        ds = ds.assign_coords(lon=np.linspace(-180, 180, lon_size))

    return ds


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%
def regrid_dataset(ds, resolution=1.0, method="nearest_s2d", verbose=False):
    """
    Regridding function.
    """
    target_grid = xe.util.grid_global(1, 1)
    regridder = xe.Regridder(
        ds, target_grid, method="nearest_s2d", periodic=True, ignore_degenerate=True
    )
    ds_regridded = regridder(ds)

    # Rename dimensions to standard names
    rename_dict = {}
    if "y" in ds_regridded.dims:
        rename_dict["y"] = "lat"
    if "x" in ds_regridded.dims:
        rename_dict["x"] = "lon"

    if rename_dict:
        ds_regridded = ds_regridded.rename(rename_dict)

    # Convert 2D coordinates to 1D
    if "lat" in ds_regridded.coords and ds_regridded.coords["lat"].ndim == 2:
        lat_values = ds_regridded.coords["lat"].isel(lon=0).values
        ds_regridded = ds_regridded.drop_vars("lat")
        ds_regridded = ds_regridded.assign_coords(lat=("lat", lat_values))

    if "lon" in ds_regridded.coords and ds_regridded.coords["lon"].ndim == 2:
        lon_values = ds_regridded.coords["lon"].isel(lat=0).values
        ds_regridded = ds_regridded.drop_vars("lon")
        ds_regridded = ds_regridded.assign_coords(lon=("lon", lon_values))

    # Time standardization
    if "time" in ds_regridded.dims:
        try:
            # Convert to standard time format
            time_vals = ds_regridded.time.values

            # Handle various time formats
            if hasattr(time_vals[0], "year"):
                # Handle cftime objects (including NoLeap, Gregorian, etc.)
                new_time = pd.to_datetime(
                    [f"{t.year}-{t.month:02d}-01" for t in time_vals]
                )
            else:
                # Handle numpy datetime64
                time_index = pd.to_datetime(time_vals)
                new_time = pd.to_datetime(
                    [f"{t.year}-{t.month:02d}-01" for t in time_index]
                )

            ds_regridded = ds_regridded.drop_vars("time")
            ds_regridded = ds_regridded.assign_coords(time=("time", new_time))

            if verbose:
                print(f"Time has been standardized to datetime64 format")

        except Exception as e:
            if verbose:
                print(f"Time standardization failed: {e}, keeping original time format")
            # Keep original time coordinates
            pass

    if verbose:
        print(f"Regridding completed, resolution: {resolution} degrees")

    return ds_regridded


# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
def process_cmip6_data(
    variable,
    experiment,
    source_id,
    project="CMIP6",
    table_id="Omon",
    resolution=1.0,
    method="nearest_s2d",
    verbose=False,
):
    """
    Main function to process CMIP6 data: merge files and regrid

    Parameters:
    -----------
    variable : str
        Variable name, e.g. 'tos', 'nbp', 'mrso'
    experiment : str
        Experiment name, e.g. 'historical'
    source_id : str
        Model name, e.g. 'ACCESS-ESM1-5'
    project : str, default 'CMIP6'
        Project name
    table_id : str, default 'Omon'
        Table ID, e.g. 'Omon', 'Lmon'
    resolution : float, default 1.0
        Target resolution (degrees)
    method : str, default 'nearest_s2d'
        Interpolation method
    verbose : bool, default False
        Whether to display detailed information

    Returns:
    --------
    bool : Whether the processing was successful
    """

    if verbose:
        print(f"Step 1: Merge files - {source_id}/{experiment}/{variable}")

    # Step 1: Merge files
    merged_ds = merge_cmip6_files(
        variable=variable,
        experiment=experiment,
        source_id=source_id,
        table_id=table_id,
        project=project,
        verbose=verbose,
    )

    if merged_ds is None:
        if verbose:
            print("Failed to merge files")
        return False

    if verbose:
        print(f"Step 2: Regrid data - {source_id}/{experiment}/{variable}")

    # Step 2: Regrid data
    try:
        regridded_ds = regrid_dataset(
            merged_ds, resolution=resolution, method=method, verbose=verbose
        )

        if verbose:
            print(f"Step 3: Save processed data - {source_id}/{experiment}/{variable}")

        # Step 3: Save processed data
        output_dir = f"/Users/hao/EXTERNAL/PROGRESS/Carbon_Recovery_and_Climate/01-data/CMIP/{project}/{experiment}/{table_id}/{variable}/{source_id}"
        os.makedirs(output_dir, exist_ok=True)

        output_file = f"{output_dir}/{variable}_{table_id}_{source_id}_{experiment}_{resolution:.0f}deg.nc"

        # Ensure correct variable name
        if regridded_ds.name != variable:
            regridded_ds.name = variable

        # Convert to Dataset to ensure variable name is saved
        if isinstance(regridded_ds, xr.DataArray):
            regridded_ds = regridded_ds.to_dataset(name=variable)

        # Save as NetCDF file
        regridded_ds.to_netcdf(output_file)

        if verbose:
            print(f"Saved to: {output_file}")
            print(f"Final dataset shape: {regridded_ds[variable].shape}")
            print(f"Variable name: {variable}")
            print(f"Dataset variables: {list(regridded_ds.data_vars.keys())}")

        # Clean up memory
        merged_ds.close()
        regridded_ds.close()

        return True

    except Exception as e:
        if verbose:
            print(f"Error during regridding or saving: {e}")

        # Clean up memory
        try:
            merged_ds.close()
        except:
            pass

        return False


In [ ]:
overlaped_models = [
    'ACCESS-ESM1-5',
    'CanESM5', 
    'CESM2',
    'CESM2-FV2',
    'CESM2-WACCM',
    'CESM2-WACCM-FV2',
    'CMCC-CM2-SR5',
    'CMCC-ESM2',
    'E3SM-1-1',
    'E3SM-1-1-ECA',
    'EC-Earth3-CC',
    'EC-Earth3-Veg-LR',
    'GFDL-ESM4',
    'GISS-E2-1-G-CC',
    'GISS-E2-2-H',
    'INM-CM4-8',
    'INM-CM5-0',
    'IPSL-CM5A2-INCA',
    'IPSL-CM6A-LR',
    'IPSL-CM6A-LR-INCA',
    'MPI-ESM-1-2-HAM',
    'MPI-ESM1-2-LR',
    'NorESM2-LM',
    'NorESM2-MM',
    'SAM0-UNICON',
    'TaiESM1'
]


In [ ]:
# Check the model running status by searching "❌"


# TOS

In [ ]:
# Define the variables
variables_list = ['tos']
table_ids      = ['Omon']
experiments_list = ['historical']
source_ids_list  = overlaped_models

# Process all combinations of variables, experiments, and source IDs
for variable in variables_list:
    for experiment in experiments_list:
        for source_id in source_ids_list:
            print(f"\nProcessing {variable} for {source_id} under {experiment} experiment")
            success = process_cmip6_data(
                variable=variable, 
                experiment=experiment, 
                source_id=source_id, 
                project='CMIP6', 
                table_id='Omon', 
                resolution=1.0,
                method='nearest_s2d', 
                verbose=False
            )
            if success:
                print(f"✅ Successfully processed {variable} for {source_id} under {experiment}")
            else:
                print(f"❌ Failed to process {variable} for {source_id} under {experiment}")



Processing tos for ACCESS-ESM1-5 under historical experiment
✅ Successfully processed tos for ACCESS-ESM1-5 under historical

Processing tos for CanESM5 under historical experiment
✅ Successfully processed tos for CanESM5 under historical

Processing tos for CESM2 under historical experiment
✅ Successfully processed tos for CESM2 under historical

Processing tos for CESM2-FV2 under historical experiment
✅ Successfully processed tos for CESM2-FV2 under historical

Processing tos for CESM2-WACCM under historical experiment
✅ Successfully processed tos for CESM2-WACCM under historical

Processing tos for CESM2-WACCM-FV2 under historical experiment
✅ Successfully processed tos for CESM2-WACCM-FV2 under historical

Processing tos for CMCC-CM2-SR5 under historical experiment
✅ Successfully processed tos for CMCC-CM2-SR5 under historical

Processing tos for CMCC-ESM2 under historical experiment
✅ Successfully processed tos for CMCC-ESM2 under historical

Processing tos for E3SM-1-1 under his

In [ ]:
ds = xr.open_dataset('/Users/hao/EXTERNAL/PROGRESS/Carbon_Recovery_and_Climate/01-data/CMIP/CMIP6/historical/Omon/tos/ACCESS-ESM1-5/tos_Omon_ACCESS-ESM1-5_historical_1deg.nc')

ds 


<xarray.Dataset> Size: 513MB
Dimensions:  (lat: 180, lon: 360, time: 1980)
Coordinates:
  * lat      (lat) float64 1kB -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
  * lon      (lon) float64 3kB -179.5 -178.5 -177.5 -176.5 ... 177.5 178.5 179.5
  * time     (time) datetime64[ns] 16kB 1850-01-01 1850-02-01 ... 2014-12-01
Data variables:
    tos      (time, lat, lon) float32 513MB ...

# NBP

In [ ]:
# Define the variables
variables_list = ['nbp']
table_ids      = ['Lmon']
experiments_list = ['historical']
source_ids_list  = overlaped_models

# Process all combinations of variables, experiments, and source IDs
for variable in variables_list:
    for experiment in experiments_list:
        for source_id in source_ids_list:
            print(f"\nProcessing {variable} for {source_id} under {experiment} experiment")
            success = process_cmip6_data(
                variable=variable, 
                experiment=experiment, 
                source_id=source_id, 
                project='CMIP6', 
                table_id='Lmon', 
                resolution=1.0,
                method='nearest_s2d', 
                verbose=False
            )
            if success:
                print(f"✅ Successfully processed {variable} for {source_id} under {experiment}")
            else:
                print(f"❌ Failed to process {variable} for {source_id} under {experiment}")



Processing nbp for ACCESS-ESM1-5 under historical experiment
✅ Successfully processed nbp for ACCESS-ESM1-5 under historical

Processing nbp for CanESM5 under historical experiment
✅ Successfully processed nbp for CanESM5 under historical

Processing nbp for CESM2 under historical experiment
✅ Successfully processed nbp for CESM2 under historical

Processing nbp for CESM2-FV2 under historical experiment
✅ Successfully processed nbp for CESM2-FV2 under historical

Processing nbp for CESM2-WACCM under historical experiment
✅ Successfully processed nbp for CESM2-WACCM under historical

Processing nbp for CESM2-WACCM-FV2 under historical experiment
✅ Successfully processed nbp for CESM2-WACCM-FV2 under historical

Processing nbp for CMCC-CM2-SR5 under historical experiment
✅ Successfully processed nbp for CMCC-CM2-SR5 under historical

Processing nbp for CMCC-ESM2 under historical experiment
✅ Successfully processed nbp for CMCC-ESM2 under historical

Processing nbp for E3SM-1-1 under his

# MRSO

In [ ]:
# Define the variables
variables_list = ['mrso']
table_ids      = ['Lmon']
experiments_list = ['historical']
source_ids_list  = overlaped_models

# Process all combinations of variables, experiments, and source IDs
for variable in variables_list:
    for experiment in experiments_list:
        for source_id in source_ids_list:
            print(f"\nProcessing {variable} for {source_id} under {experiment} experiment")
            success = process_cmip6_data(
                variable=variable, 
                experiment=experiment, 
                source_id=source_id, 
                project='CMIP6', 
                table_id='Lmon', 
                resolution=1.0,
                method='nearest_s2d', 
                verbose=False
            )
            if success:
                print(f"✅ Successfully processed {variable} for {source_id} under {experiment}")
            else:
                print(f"❌ Failed to process {variable} for {source_id} under {experiment}")



Processing mrso for ACCESS-ESM1-5 under historical experiment
✅ Successfully processed mrso for ACCESS-ESM1-5 under historical

Processing mrso for CanESM5 under historical experiment
✅ Successfully processed mrso for CanESM5 under historical

Processing mrso for CESM2 under historical experiment
✅ Successfully processed mrso for CESM2 under historical

Processing mrso for CESM2-FV2 under historical experiment
✅ Successfully processed mrso for CESM2-FV2 under historical

Processing mrso for CESM2-WACCM under historical experiment
✅ Successfully processed mrso for CESM2-WACCM under historical

Processing mrso for CESM2-WACCM-FV2 under historical experiment
✅ Successfully processed mrso for CESM2-WACCM-FV2 under historical

Processing mrso for CMCC-CM2-SR5 under historical experiment
✅ Successfully processed mrso for CMCC-CM2-SR5 under historical

Processing mrso for CMCC-ESM2 under historical experiment
✅ Successfully processed mrso for CMCC-ESM2 under historical

Processing mrso for E

# fgCO2

In [ ]:
# Define the variables
variables_list = ['fgco2']
table_ids      = ['Omon']
experiments_list = ['historical']
source_ids_list  = overlaped_models

# Process all combinations of variables, experiments, and source IDs
for variable in variables_list:
    for experiment in experiments_list:
        for source_id in source_ids_list:
            print(f"\nProcessing {variable} for {source_id} under {experiment} experiment")
            success = process_cmip6_data(
                variable=variable, 
                experiment=experiment, 
                source_id=source_id, 
                project='CMIP6', 
                table_id='Omon', 
                resolution=1.0,
                method='nearest_s2d', 
                verbose=False
            )
            if success:
                print(f"✅ Successfully processed {variable} for {source_id} under {experiment}")
            else:
                print(f"❌ Failed to process {variable} for {source_id} under {experiment}")



Processing fgco2 for NorESM2-LM under historical experiment
✅ Successfully processed fgco2 for NorESM2-LM under historical
